The Ultimate Python Scope & Memory Example

In [1]:
def master_outer(o_num, o_dict, o_def_num=10, o_def_list=[]):
    # --- OUTER LOCALS ---
    o_loc_str = "O_Start"  # Outer Local Immutable (String)
    o_loc_list = ["O_Loc"]  # Outer Local Mutable (List)

    # --- INNER FUNCTION DEFINITION ---
    # Includes: non-default immutable (i_num), non-default mutable (i_dict)
    #           default immutable (i_def_num), default mutable (i_def_list)
    def master_inner(i_num, i_dict, i_def_num=99, i_def_list=[]):

        # --- PERMISSION TO REASSIGN OUTER IMMUTABLES ---
        # Strings and numbers are locked. We need the master key to change their addresses.
        nonlocal o_loc_str, o_def_num

        # 1. Mutating Outer Variables (Closure Backpack)
        # We reach into the Heap and change the contents. No 'nonlocal' needed.
        o_dict["calls"] = o_dict.get("calls", 0) + 1
        o_def_list.append(o_num)
        o_loc_list.append("O_Mutated")

        # 2. Reassigning Outer Immutables (Closure Backpack)
        # We cross out the old sticky note addresses and write new ones.
        o_loc_str = o_loc_str + "!"
        o_def_num = o_def_num + 1

        # 3. Inner Locals
        i_loc_str = "I_Start"  # Inner Local Immutable
        i_loc_list = ["I_Loc"]  # Inner Local Mutable

        # 4. Mutating & Reassigning Inner Variables
        # These are local to this specific function run, so we can do whatever we want.
        i_dict["status"] = "active"
        i_def_list.append(i_num)
        i_loc_list.append("I_Mutated")
        i_loc_str = i_loc_str + "!"
        i_def_num = i_def_num + 1

        return {
            "Outer_Memory_State": {
                "o_dict": o_dict,  # Mutated outer parameter
                "o_def_list": o_def_list,  # Mutated outer default
                "o_loc_list": o_loc_list,  # Mutated outer local
                "o_loc_str": o_loc_str,  # Reassigned outer local string
                "o_def_num": o_def_num,  # Reassigned outer default number
            },
            "Inner_Memory_State": {
                "i_dict": i_dict,  # Mutated inner parameter
                "i_def_list": i_def_list,  # Mutated inner default
                "i_loc_list": i_loc_list,  # Mutated inner local
                "i_loc_str": i_loc_str,  # Reassigned inner local string
                "i_def_num": i_def_num,  # Reassigned inner default number
            },
        }

    return master_inner

Step-by-Step Memory Breakdown
Phase 1: Script Execution (The Global Setup)
Before you even call a function, Python reads def master_outer... and builds it.

Outer Defaults Created: Python creates the integer 10 and the list [] in the Heap.

Storage: It saves their memory addresses in master_outer.__defaults__.

Note: The inner function does NOT exist yet.

Phase 2: Calling master_outer (Creating the Closures)
Let's make two separate closures:

In [2]:
my_outer_dict_1 = {}
func_A = master_outer(o_num=1, o_dict=my_outer_dict_1)

my_outer_dict_2 = {}
func_B = master_outer(o_num=2, o_dict=my_outer_dict_2)
# we are using 2 diff dicts to show that the inner functions are not sharing the same outer mutable parameter. They have their own closure backpacks with their own copies of the outer variables.:
# but if we have used the same dict for both, then they would have shared it and we would have seen the mutations from both inner functions in the same dict.

What happened behind the scenes for func_A and func_B?

Shared Outer Defaults: Because we didn't pass custom lists or numbers for o_def_list and o_def_num, both func_A and func_B grab the exact same addresses from master_outer.__defaults__. They are secretly sharing memory.

Proof:

In [3]:
for cell in func_A.__closure__:
    print(f" {cell.cell_contents}: {id(cell.cell_contents)}")

 []: 2104905034368
 10: 140720408294808
 {}: 2104905393088
 ['O_Loc']: 2104905296448
 O_Start: 2104904966784
 1: 140720408294520


In [4]:
for cell in func_B.__closure__:
    print(f" {cell.cell_contents}: {id(cell.cell_contents)}")

 []: 2104905034368
 10: 140720408294808
 {}: 2104905393216
 ['O_Loc']: 2104905397888
 O_Start: 2104904966784
 2: 140720408294552


Isolated Outer Locals: Every time you call master_outer, it creates fresh local variables. func_A has its own o_loc_str ("O_Start"), and func_B has its own completely separate o_loc_str. and same with o_loc_list

1. o_loc_list (The Lists)
Yes, they are completely separate. Your above code proves it: the IDs for func_A and for func_B are different . Every time the outer function runs, it creates a brand new list in the Heap.

2. O_Start (The String)
Your intuition is 100% correct. This is just Python being highly memory efficient through a process called String Interning.

Because strings are immutable, Python knows it is safe to reuse them. Instead of wasting memory building two identical "O_Start" houses, Python builds it once and points both local sticky notes to the exact same address.

both o_def_num (the immutable 10) and o_def_list (the mutable []) start out exactly the same way: they are both built once, and both func_A and func_B start out pointing to the exact same memory addresses for them.

But here is the crucial reason why their behavior splits later on. It all comes down to what happens when you try to change them.

1. The Mutable Shared Link (o_def_list)
Because it is a list, func_A goes to the shared address and puts an item inside. It does not cross out the address in its closure cell.
Because func_A kept the original address, and func_B still has the original address, they continue to share the exact same list forever.

2. The Immutable Broken Link (o_def_num = 10)
You are right that func_A and func_B both start with closures pointing to the exact same 10.

But watch what happens when func_A tries to change it:

func_A runs: o_def_num = o_def_num + 1.

Because 10 is immutable, Python builds an 11 somewhere else.

Because of nonlocal, func_A crosses out the 10 address in its closure cell and writes down the address for 11.

The Link is Broken: Now func_A points to 11. But what about func_B? func_B's closure cell was untouched! It still happily points to the original 10.

Summary
Your core reasoning is flawless: they both start by pointing to the exact same memory because they are outer defaults.

The only difference is that o_def_list keeps its address (so they stay linked), while o_def_num is forced to change its address the moment you do math on it (which breaks the shared link between func_A and func_B).

Inner Defaults are Created NOW: When master_outer runs, it executes the def master_inner... block. This means func_A builds a brand new default list [] for i_def_list. Then, when master_outer runs again for func_B, it builds a second, completely separate default list [] for func_B's i_def_list.

Proof:

In [5]:
func_A.__defaults__

(99, [])

In [6]:
func_B.__defaults__

(99, [])

In [7]:
type(func_B.__defaults__)

tuple

In [8]:
id(func_A.__defaults__[0]), id(func_B.__defaults__[0])

(140720408297656, 140720408297656)

In [9]:
id(func_A.__defaults__[1]), id(func_B.__defaults__[1])

(2104905296512, 2104905349696)

1. You are right about the Memory Efficiency (Integer Caching)
Your screenshot is checking __defaults__[0], which corresponds to i_def_num=99.

Because func_A and func_B are two completely different function objects, they each have their own separate __defaults__ tuple. However, as you brilliantly deduced, Python is being highly memory-efficient. When it evaluated 99 for func_A, it used the cached 99. When it evaluated 99 for func_B, it used the exact same cached 99.

So, yes, both tuples are pointing to the exact same immutable integer object in the Heap!

2. The Correction: Will the ID in __defaults__ change after calling?
You asked: "and they wil change once we call any one inner fn to change its variablkes?"

No. The ID inside __defaults__ will NEVER change.

This is a massive point to understand. The __defaults__ attribute is a permanent, read-only record created the moment the def statement finishes.

When you run func_A() and the code hits this line:
i_def_num = i_def_num + 1

Here is exactly what happens:

Python creates a temporary Stack Frame for the call.

It creates a temporary local sticky note named i_def_num.

Because you didn't pass an argument, it copies the address from __defaults__[0] (which points to 99) onto the local sticky note.

It does 99 + 1 = 100.

It crosses out the address on the temporary local sticky note and writes the address for 100.

The __defaults__ tuple is untouched. It still happily points to 99.



What about the List (__defaults__[1])?
If you check id(func_A.__defaults__[1]) and id(func_B.__defaults__[1]), you will find they are DIFFERENT addresses.

Why? Because Python does not cache lists. Every time def master_inner... runs, [] physically builds a brand new house in the Heap. So func_A gets its own private list default, and func_B gets its own private list default.

And even when you mutate that list by doing i_def_list.append(100), the ID stored in __defaults__[1] still does not change. It still points to the exact same house; the house just has more furniture inside it now.

Summary
Immutable Defaults (99): Start with the same ID due to caching. The ID in __defaults__ never changes. Reassigning only changes the temporary local variable.

Mutable Defaults ([]): Start with completely different IDs because def creates a new list every time the outer function runs. The ID in __defaults__ never changes, but the contents of that list do change when you append to it.



The Ultimate Proof
If you run this code after calling func_A, you will see that __defaults__ is permanently locked.


In [10]:
# Call the function. This reassigns the local variable to 100.
my_inner_dict_1 = {}
print(func_A(i_num=100, i_dict=my_inner_dict_1))

# Check the defaults again...
print("After calling:")
print(
    f"func_A default[0] ID: {id(func_A.__defaults__[0])}"
)  # Exactly the same ID as before!
print(f"func_A default[0] Value: {func_A.__defaults__[0]}")  # Still 99!

{'Outer_Memory_State': {'o_dict': {'calls': 1}, 'o_def_list': [1], 'o_loc_list': ['O_Loc', 'O_Mutated'], 'o_loc_str': 'O_Start!', 'o_def_num': 11}, 'Inner_Memory_State': {'i_dict': {'status': 'active'}, 'i_def_list': [100], 'i_loc_list': ['I_Loc', 'I_Mutated'], 'i_loc_str': 'I_Start!', 'i_def_num': 100}}
After calling:
func_A default[0] ID: 140720408297656
func_A default[0] Value: 99


# i_def_num here prints 100 as it is returned after changinf locally for klocal varioable: but actual default is set to 99: immurtable fix vlue in default from func definition

The Golden Rules for Your Notes
Evaluated Only Once: Python evaluates the default argument 0 exactly once when the def statement runs and saves its memory address inside __defaults__.

The = Sign (Reassignment): The line total = total + points evaluates the math, creates a brand new integer in the Heap, and writes that new address onto the local total variable.

The Result: Because we only updated the local variable, the address safely stored inside __defaults__ is never crossed out. It permanently points to the original 0 for every future function call.

In [11]:
def add_score(points, total=0):
    print(f"  [Inside] Start ID of 'total': {id(total)}")

    # THE REASSIGNMENT
    total = total + points

    print(f"  [Inside] End ID of 'total':   {id(total)} <-- It moved!")
    return total


print("--- IMMUTABLE DEFAULT PROOF ---")
# 1. Print the ID of the default stored in the Heap
print(f"Default ID in Heap: {id(add_score.__defaults__[0])}")

print("\nCall 1:")
add_score(5)

print("\nCall 2:")
add_score(10)

# 2. Prove the default in the Heap was never touched
print(f"\nFinal Default ID:   {id(add_score.__defaults__[0])} <-- Untouched!")

--- IMMUTABLE DEFAULT PROOF ---
Default ID in Heap: 140720408294488

Call 1:
  [Inside] Start ID of 'total': 140720408294488
  [Inside] End ID of 'total':   140720408294648 <-- It moved!

Call 2:
  [Inside] Start ID of 'total': 140720408294488
  [Inside] End ID of 'total':   140720408294808 <-- It moved!

Final Default ID:   140720408294488 <-- Untouched!


In [12]:
# after caaling func_A:
for cell in func_A.__closure__:
    print(f" {cell.cell_contents}: {id(cell.cell_contents)}")

 [1]: 2104905034368
 11: 140720408294840
 {'calls': 1}: 2104905393088
 ['O_Loc', 'O_Mutated']: 2104905296448
 O_Start!: 2104905296048
 1: 140720408294520


In [13]:
# without caaling func_B as of now:
for cell in func_B.__closure__:
    print(f" {cell.cell_contents}: {id(cell.cell_contents)}")

 [1]: 2104905034368
 10: 140720408294808
 {}: 2104905393216
 ['O_Loc']: 2104905397888
 O_Start: 2104904966784
 2: 140720408294552


In [14]:
for default in func_A.__defaults__:
    print(f"func_A default value: {default}, ID: {id(default)}")

func_A default value: 99, ID: 140720408297656
func_A default value: [100], ID: 2104905296512


In [15]:
for default in func_B.__defaults__:
    print(f"func_B default value: {default}, ID: {id(default)}")

func_B default value: 99, ID: 140720408297656
func_B default value: [], ID: 2104905349696


Outer Mutable (o_def_list): Goes to the shared Heap address and appends 1. (List is [1]).

Outer Immutable (o_def_num): Uses nonlocal to cross out 10 and write 11.

Inner Mutable (i_def_list): Goes to func_A's personal default list and appends 100. (List is [100]).

Run 2: Hitting func_A AGAIN

In [16]:
my_inner_dict_2 = {}
print(func_A(i_num=200, i_dict=my_inner_dict_2))

{'Outer_Memory_State': {'o_dict': {'calls': 2}, 'o_def_list': [1, 1], 'o_loc_list': ['O_Loc', 'O_Mutated', 'O_Mutated'], 'o_loc_str': 'O_Start!!', 'o_def_num': 12}, 'Inner_Memory_State': {'i_dict': {'status': 'active'}, 'i_def_list': [100, 200], 'i_loc_list': ['I_Loc', 'I_Mutated'], 'i_loc_str': 'I_Start!', 'i_def_num': 100}}


Outer Mutable (o_def_list): Goes back to the shared Heap address. Appends 1 again. (List is [1, 1]).

Outer Immutable (o_def_num): Crosses out 11 and writes 12.

Inner Mutable (i_def_list): Because this is the exact same func_A object, it uses the exact same inner default list. It appends 200. (List is [100, 200]).

Inner Locals: Variables like i_loc_str and i_loc_list are wiped out and recreated fresh on every call. They never accumulate data.

Run 3: Hitting func_B (The Proof of Isolation and Sharing)

In [17]:
my_inner_dict_3 = {}
print(func_B(i_num=300, i_dict=my_inner_dict_3))

{'Outer_Memory_State': {'o_dict': {'calls': 1}, 'o_def_list': [1, 1, 2], 'o_loc_list': ['O_Loc', 'O_Mutated'], 'o_loc_str': 'O_Start!', 'o_def_num': 11}, 'Inner_Memory_State': {'i_dict': {'status': 'active'}, 'i_def_list': [300], 'i_loc_list': ['I_Loc', 'I_Mutated'], 'i_loc_str': 'I_Start!', 'i_def_num': 100}}


Outer Mutable (o_def_list): It goes to the shared Heap address! It appends 2. (The shared list is now [1, 1, 2]).

Outer Immutable (o_def_num): Wait, func_B shares the same __defaults__ address (which was 10). But remember, func_A crossed out its sticky note to point to 12. func_B's sticky note still points to 10. So func_B crosses out 10 and writes 11.

Inner Mutable (i_def_list): func_B has its own isolated inner default list. It appends 300. (List is just [300], completely ignoring func_A's [100, 200]).

In [18]:
my_inner_dict_4 = {}
print(func_B(i_num=400, i_dict=my_inner_dict_4))

{'Outer_Memory_State': {'o_dict': {'calls': 2}, 'o_def_list': [1, 1, 2, 2], 'o_loc_list': ['O_Loc', 'O_Mutated', 'O_Mutated'], 'o_loc_str': 'O_Start!!', 'o_def_num': 12}, 'Inner_Memory_State': {'i_dict': {'status': 'active'}, 'i_def_list': [300, 400], 'i_loc_list': ['I_Loc', 'I_Mutated'], 'i_loc_str': 'I_Start!', 'i_def_num': 100}}


The 4 Absolute Rules of Python Memory

If you internalize these four rules, you will never get confused by Python scope again:



Immutables (Strings/Numbers/Tuples) are locked. You cannot use an = sign on them inside an inner function unless you declare nonlocal. If you don't use nonlocal, Python creates a brand new local variable instead, which often causes a crash if used like this: x= x+1 or x= x + "something".

Mutables (Lists/Dicts/Sets) are unlocked. You can reach inside them and .append() or mutate them anywhere, anytime, without needing nonlocal.

Outer Defaults (def outer(x=[])) are shared across all closures. Every inner function you generate will step on each other's toes if they mutate that list.

Inner Defaults (def inner(x=[])) are shared ONLY within that specific closure. If you call func_A ten times, it shares the list. But func_B gets its own separate sandbox.

Using an = sign on its own does not cause a crash. It is only when you try to read and reassign at the same time (like x = x + 1) that the crash happens.

Here is the exact distinction you perfectly pointed out:

1. Pure Assignment (No Crash)
If you just do x = 5 or x = "new string", Python does not panic. It simply says, "Okay, the programmer wants a brand new local sticky note named x." It completely ignores the outer variable, creates the new local one, and the code runs perfectly.

In [19]:
def outer():
    x = "Outer String"

    def inner():
        x = "Brand New Local String"  # Pure assignment. NO CRASH.
        print("Inside:", x)

    inner()
    print("Outside:", x)  # The outer string was completely untouched.


outer()

Inside: Brand New Local String
Outside: Outer String


2. Read-and-Reassign (The Crash)
If you do x = x + 1 or x = x + "something", this is where Python crashes with the UnboundLocalError.

As you correctly remembered from our parsing discussion:

Python sees the = sign and makes x a local variable.

It tries to evaluate the right side (x + 1).

It looks at the local sticky note for x, sees that it is totally blank, and crashes because it cannot add 1 to nothing.

In [ ]:
def outer():
    x = 10

    def inner():
        x = (
            x + 1
        )  # CRASH! UnboundLocalError: local variable 'x' referenced before assignment

    inner()

In [22]:
# outer()
# crash

In [24]:
func_A.__closure__

(<cell at 0x000001EA161CDDE0: list object at 0x000001EA161DEA80>,
 <cell at 0x000001EA161CF610: int object at 0x00007FFC05F2F5D8>,
 <cell at 0x000001EA161CD840: dict object at 0x000001EA162363C0>,
 <cell at 0x000001EA161CF580: list object at 0x000001EA1621EA40>,
 <cell at 0x000001EA161CFDC0: str object at 0x000001EA1621FD30>,
 <cell at 0x000001EA1618DD80: int object at 0x00007FFC05F2F478>)

In [25]:
for cell in func_A.__closure__:
    print(f" {cell.cell_contents}: {id(cell.cell_contents)}")

 [1, 1, 2, 2]: 2104905034368
 12: 140720408294872
 {'calls': 2}: 2104905393088
 ['O_Loc', 'O_Mutated', 'O_Mutated']: 2104905296448
 O_Start!!: 2104905301296
 1: 140720408294520


In [26]:
for cell in func_A.__closure__:
    print(f" {cell}: {id(cell)}")

 <cell at 0x000001EA161CDDE0: list object at 0x000001EA161DEA80>: 2104904965600
 <cell at 0x000001EA161CF610: int object at 0x00007FFC05F2F5D8>: 2104904971792
 <cell at 0x000001EA161CD840: dict object at 0x000001EA162363C0>: 2104904964160
 <cell at 0x000001EA161CF580: list object at 0x000001EA1621EA40>: 2104904971648
 <cell at 0x000001EA161CFDC0: str object at 0x000001EA1621FD30>: 2104904973760
 <cell at 0x000001EA1618DD80: int object at 0x00007FFC05F2F478>: 2104904703360


In [29]:
int("0x000001EA161CDDE0", 16)

2104904965600

In [32]:
for i, cell in enumerate(func_A.__closure__):
    print(f"{i}: {cell}: {id(cell)}")

0: <cell at 0x000001EA161CDDE0: list object at 0x000001EA161DEA80>: 2104904965600
1: <cell at 0x000001EA161CF610: int object at 0x00007FFC05F2F5D8>: 2104904971792
2: <cell at 0x000001EA161CD840: dict object at 0x000001EA162363C0>: 2104904964160
3: <cell at 0x000001EA161CF580: list object at 0x000001EA1621EA40>: 2104904971648
4: <cell at 0x000001EA161CFDC0: str object at 0x000001EA1621FD30>: 2104904973760
5: <cell at 0x000001EA1618DD80: int object at 0x00007FFC05F2F478>: 2104904703360
